In [2]:
# from ipysigma import Sigma
import networkx as nx
import pandas as pd
from rapidfuzz import fuzz, process, utils

### Clean Data

In [3]:
network_df = pd.read_csv("../data/processed/combined_entities.csv", index_col=0)
network_df.reset_index(inplace=True)
network_df.rename(columns={"index": "report_entity_no"}, inplace=True)

In [ ]:
network_df.info()

### Edit individuals

Aim to increase non-null coverage for heading_survey_area, heading_survey_party to 100%, and to correct errors for individuals found during cleaning

In [ ]:
# 5 variable records for Baboo M. S. Dutt
network_df.loc[network_df.query("person == 'Baboo M. S. Dutt'").index, "title"] = "Baboo"
network_df.loc[network_df.query("person == 'Baboo M. S. Dutt'").index, "firstname"] = "M. S."
network_df.loc[network_df.query("person == 'Baboo M. S. Dutt'").index, "lastname"] = "Dutt"

# Mislabelled title for J. Harper
network_df.loc[network_df.query("title == 'Harper'").index, "lastname"] = "Harper"
network_df.loc[network_df.query("title == 'Harper'").index, "title"] = "Mr"

# Mislabelled title for Gour Chundra
network_df.loc[network_df.query("title == 'Gour Chundra'").index, "firstname"] = "Gour"
network_df.loc[network_df.query("title == 'Gour Chundra'").index, "lastname"] = "Chundra"
network_df.loc[network_df.query("title == 'Gour Chundra'").index, "title"] = "Native Surveyor"

# S. Cooper is actually A. Cooper
network_df.loc[network_df.query("person == 'Mr. S. Cooper'").index, "firstname"] = "A."
network_df.loc[network_df.query("person == 'Mr. S. Cooper'").index, "person"] = "Mr. A. Cooper"

# standardise the Strahans

# Standardise Moung Keng as surname
network_df.loc[network_df["person"].str.contains("Keng"), "lastname"] = "Moung Keng"

In [ ]:
# 1 ,7 survey party individuals from 1865 survey all worked on the Gwalior and Central India survey that year
network_df.loc[network_df.query(
    "heading_survey_party == 'No. 1 TOPOGRAPHICAL PARTY, No. 7 TOPOGRAPHICAL PARTY'"
).index, "heading_survey_area"] = "GWALIOR AND CENTRAL INDIA SURVEY"

# 1, 2 and 7 survey parties all present though
network_df.loc[network_df.query(
    "heading_survey_party == 'No. 1 TOPOGRAPHICAL PARTY, No. 7 TOPOGRAPHICAL PARTY'"
).index, "heading_survey_party"] = ["1"] * 9 + ["2"] * 3 + ["7"] * 3

# 
network_df.loc[network_df.query(
    "(heading_survey_party == 'No. 3 TOPOGRAPHIICAL PARTY' or heading_survey_party == 'No. III PARTY') and report_date == 1867"
).index, "heading_survey_area"] = "CENTRAL PROVINCES AND VIZAGAPATAM AGENCY"

In [ ]:
# Survey area in 1866 survey for 1/7 party is mis-parsed due to double survey area
network_df.loc[(network_df["report_date"] == 1866) & (network_df["heading_survey_area"].isna()), "heading_survey_area"] = "GWALIOR AND CENTRAL INDIA, RAJPOOTANA"

# Survey party 1/7 in 1866 survey is mis-parsed due to double survey party
network_df.loc[(network_df["report_date"] == 1866) & (network_df["heading_survey_party"].isna()), "heading_survey_party"] = "1"
network_df.loc[
    (network_df["report_date"] == 1866) 
    & (network_df["heading_survey_party"] == "1")
    & ((11 < network_df["report_entity_no"]) & (network_df["report_entity_no"] < 17) | (network_df["report_entity_no"] == 21))
    , "heading_survey_party"] = "7"

In [ ]:
# Survey area in 1869 survey for party 1 is mis-parsed due to no credit area
network_df.loc[(network_df["report_date"] == 1869) & (network_df["heading_survey_area"].isna()), "heading_survey_area"] = "GWALIOR AND CENTRAL INDIA"

# Survey party 1 in 1869 survey is mis-parsed due to double survey party
network_df.loc[(network_df["report_date"] == 1869) & (network_df["heading_survey_party"].isna()), "heading_survey_party"] = "1"

In [ ]:
# Survey party 2 in 1867 survey doesn't have heading_survey_area as it is pulled from narrative report, not initial credit
# This is because initial credit is 4 regions after heading due to reading order
network_df.loc[
    (network_df["report_date"] == 1867)
    & (network_df["heading_survey_area"].isna())
    & (network_df["heading_survey_party"] == "No. 2 TOPOGRAPHICAL PARTY"),
    "heading_survey_area"] = "HYDERABAD"

# Second mentions of survey party 6 in 1867 survey doesn't have heading_survey_area as it is pulled from narrative report, not initial credit
# Initial credit has already been parsed in this case
network_df.loc[
    (network_df["report_date"] == 1867)
    & (network_df["heading_survey_area"].isna())
    & (network_df["heading_survey_party"] == "No. 6 TOPOGRAPHICAL SURVEY PARTY"),
    "heading_survey_area"] = "KHASIA AND GARO HILLS"

In [ ]:
assert (~network_df["heading_survey_area"].isna()).all()
assert (~network_df["heading_survey_party"].isna()).all()
network_df.info()

### Clean columns

In [ ]:
def combine_str_na(firstname, lastname):
    if pd.isna(firstname):
        firstname = ""
    if pd.isna(lastname):
        lastname = ""
    if firstname or lastname:
        return (firstname + " " + lastname).strip(" ")
    else:
        return pd.NA

In [ ]:
network_df["person"] = network_df["person"].str.replace("\xa0", " ")
network_df["person"] = network_df["person"].str.replace("\n", " ")
network_df["person"] = network_df["person"].str.strip(".").str.strip(" ")
network_df["person"] = network_df["person"].str.strip("(").str.strip(")")

In [ ]:
network_df.loc[network_df["ethnicity"].isna(), "ethnicity"] = "Non-native"

In [ ]:
network_df.insert(4, "fullname", network_df.apply(lambda x: combine_str_na(x["firstname"], x["lastname"]), axis=1))
network_df["fullname"] = network_df["fullname"].str.strip(".").str.strip(" ")
network_df.insert(5, "unique_id", -1)

In [ ]:
network_df.info()

#### Clean survey area + party

In [ ]:
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.upper().str.strip(".").str.strip("'")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace("  ", " ")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace("THE ", "")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace(" SURVEY", "")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace(", DIVISION", "")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace(" DIVISION", "")
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('HYDRABAD', 'HYDERABAD')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('CENTRAL PROVINCES AND VIZAGAPATAM, AGENCY', 'CENTRAL PROVINCES AND VIZAGAPATAM AGENCY')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('CENTRAL PROVINCES AND VIZAGAPATAM\nAGENCY', 'CENTRAL PROVINCES AND VIZAGAPATAM AGENCY')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('CENTRAL PROVINCES AND VIZAGAPATAM AGENCY', 'CENTRAL PROVINCES AND VIZAGAPATAM AGENCY')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('COSSYAH AND GARROW HILLS', 'KHASIA AND GARO HILLS')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('KOSSIA AND GARROW HILLS', 'KHASIA AND GARO HILLS')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('KHASIA AND GARROW HILLS', 'KHASIA AND GARO HILLS')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('BUNDELCUND', 'BUNDELKUND')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('REWAH AND BUNDELCUND', 'REWAH, BUNDELKUND')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('REWAH AND BUNDELKUND', 'REWAH, BUNDELKUND')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('REWAH TERRITORY', 'REWAH')

# network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('RAJPOOTANA', 'RAJPOOTANA')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('CENTRAL PROVINCES AND VIZAGAPATAM AGENCY', 'CENTRAL PROVINCES, VIZAGAPATAM AGENCY')
network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('CENTRAL PROVINCES, LATE HYDERABAD', 'CENTRAL PROVINCES, HYDERABAD')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace('PROVINES', 'PROVINCES')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.replace(', AND', ',')

network_df["heading_survey_area"] = network_df["heading_survey_area"].str.strip().str.strip("'")

In [ ]:
network_df["heading_survey_area"].unique()

In [ ]:
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.upper()
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.strip(".").str.strip(",")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace(" SURVEY", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace(" PARTY", "")

network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. III', "3")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. IV', "4")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. VII', "7")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. VI', "6")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. V', "5")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace('NO. I', "1")

network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace("NO. ", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace("TOPOGRAPHICAL", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace("TOPOGRAPHIICAL", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace(".—", "")

network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace("THE ", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace("PEGU", "8")

network_df["heading_survey_party"] = network_df["heading_survey_party"].str.replace(", ", "")
network_df["heading_survey_party"] = network_df["heading_survey_party"].str.strip()

In [ ]:
network_df["heading_survey_party"].unique()

#### Remove invalid entities

In [ ]:
complete_network_df = network_df.copy()

In [ ]:
invalid_entities = network_df.loc[(network_df["lastname"].isna()) & (network_df["person"].str.contains("Native Surveyors")), :].index
invalid_entities = invalid_entities.append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["person"] == "Sub-Assistant"), :].index
).append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["person"] == "4th Grade Sub-Assistant"), :].index
).append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["person"] == "4th grade"), :].index
).append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["person"] == "Sub-Assistant, 4th grade"), :].index
).append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["person"] == "Assistant Surveyor"), :].index
).append(
    network_df.loc[(network_df["lastname"].isna()) & (network_df["heading_survey_area"] == "KHASIA AND GARO HILLS"), :].index
)

In [ ]:
network_df.drop(index=invalid_entities, inplace=True)

In [ ]:
network_df.dropna(subset="fullname", inplace=True)

### Name match people

People who are definitely ambiguous:  
Mr. Kitchen  
Mr. Chennell

In [ ]:
fuzzy_gt_raw_df = pd.read_csv("../data/processed/combined_entities_entity_deduplication.csv", index_col=0, encoding="utf-8-sig")
fuzzy_gt_raw_df.reset_index(inplace=True)
fuzzy_gt_raw_df["person"] = fuzzy_gt_raw_df["person"].str.replace("\xa0", " ")
fuzzy_gt_raw_df["fullname"] = fuzzy_gt_raw_df.apply(lambda x: combine_str_na(x["firstname"], x["lastname"]), axis=1)

In [ ]:
fuzzy_gt_df = fuzzy_gt_raw_df.dropna(subset="exists_in_row").copy()
fuzzy_gt_df.loc[:,"exists_in_row"] = "D" + (fuzzy_gt_df.index + 2).astype(str) + ", " + fuzzy_gt_df["exists_in_row"]
fuzzy_gt_df.set_index("fullname", inplace=True)
fuzzy_gt_df.drop(index="Kitchen", inplace=True)

In [ ]:
fuzzy_gt_df["exists_in_row"] = fuzzy_gt_df["exists_in_row"].str.strip(", ").str.replace("D", "").str.split(", ").apply(lambda x: sorted(list(set([int(y) - 2 for y in x]))))
fuzzy_gt = fuzzy_gt_df[["exists_in_row", "ethnicity"]]
fuzzy_gt

In [ ]:
complete_network_df.loc[fuzzy_gt.loc["T. Claudius", "exists_in_row"], :]

In [ ]:
fuzzy_match_names_raw = {}
for i, name in enumerate(network_df["fullname"].unique()):
    token_set_ratio = process.extract(name, choices=network_df["fullname"].unique()[i + 1:], limit=None, score_cutoff=75, scorer=fuzz.token_set_ratio, processor=utils.default_process)
    ratio = process.extract(name, choices=network_df["fullname"].unique()[i + 1:], limit=None, score_cutoff=75, scorer=fuzz.ratio, processor=utils.default_process)
    fuzzy_match_names_raw[name] = token_set_ratio + ratio

In [ ]:
# fuzzy_match_names = {k:v for k,v in fuzzy_match_names_raw.items() if v}
fuzzy_match_names = fuzzy_match_names_raw

In [ ]:
len(fuzzy_match_names_raw), len(fuzzy_match_names)

In [ ]:
name_matches_df = pd.DataFrame.from_dict(fuzzy_match_names, orient="index")

In [ ]:
matched_check = pd.Series(index=name_matches_df.index, data=False)
name_sets = {}
unique_idx = 0

for name, row in name_matches_df.dropna(how="all").iterrows():
    if matched_check.loc[name]:
        continue
        
    ethnicity = complete_network_df.drop_duplicates(subset="fullname").query(f"fullname == '{name}'")["ethnicity"].values[0]
    if ethnicity == "Native":
        threshold = 80  # spelling variance for native surveyor names
    else:
        threshold = 90
    
    full_matches = [x[0] for x in row.dropna() if x[1] > threshold]
    for fm in full_matches:
        if fm in name_matches_df.index:
            other_matches = [x[0] for x in name_matches_df.loc[fm].dropna() if x[1] > threshold]
            for n in other_matches:
                if n not in full_matches:
                    full_matches += [n]
    full_matches += [name]
    full_matches.sort()
    full_matches = [m for m in full_matches if not matched_check.loc[m]]
    for fm in full_matches:
        matched_check.loc[fm] = True
    name_sets[unique_idx] = full_matches
    unique_idx += 1

In [ ]:
for idx, names in name_sets.items():
    for name in names:
        network_df.loc[network_df["fullname"] == name, "unique_id"] = idx

In [ ]:
unique_idx

In [ ]:
for name, row in network_df.iterrows():
    if row["unique_id"] == -1:
        network_df.loc[name, "unique_id"] = unique_idx
        unique_idx += 1

In [ ]:
network_df["unique_id"]

### Compare name matching algorithms with ground truth

In [ ]:
algo_comparison = {}
algo_comparison_dfs = {}

for (name, row) in fuzzy_gt.iterrows():
    algo_comparison[name] = {}
    algo_comparison_dfs[name] = {}

    if row["ethnicity"] == "Native":
        threshold = 80  # spelling variance for native surveyor names
    else:
        threshold = 90
    
    full_matches = [x[0] for x in name_matches_df.loc[name].dropna() if x[1] > threshold]
    for fm in full_matches:
        if fm in name_matches_df.index:
            other_matches = [x[0] for x in name_matches_df.loc[fm].dropna() if x[1] > threshold]
            for n in other_matches:
                if n not in full_matches:
                    full_matches += [n]
    full_matches += [name]
    full_matches.sort()
    full_matches

    algo_comparison[name]["n_gt_matches"] = len(fuzzy_gt.loc[name, "exists_in_row"])
    algo_comparison_dfs[name]["gt_matches"] = fuzzy_gt.loc[name, "exists_in_row"]

    matches_to_one_name = [network_df["fullname"] == match for match in full_matches]

    algorithm_matched_names = network_df[pd.concat(matches_to_one_name, axis=1).sum(axis=1).astype(bool)].loc[:, ["person", "fullname", "report_date", "heading_survey_party", "heading_survey_area"]]
    algo_comparison[name]["n_algo_matches"] = algorithm_matched_names.shape[0]
    algo_comparison_dfs[name]["algo_matches"] = algorithm_matched_names.index.values
    algo_comparison_dfs[name]["algo_match_df"] = algorithm_matched_names

In [ ]:
complete_network_df.query("fullname == 'Abdulrahim'").loc[:, :"heading_survey_party"]

In [ ]:
complete_network_df.loc[algo_comparison_dfs["Andrew Chamarett"]["algo_matches"], :"heading_survey_party"]

In [ ]:
name_matches_df.loc["Andrew Chamarett"]

In [ ]:
compare_df = pd.DataFrame(algo_comparison).T
compare_df["diff"] = compare_df["n_gt_matches"] - compare_df["n_algo_matches"]
compare_df.describe()

In [ ]:
compare_df

In [ ]:
# > 90

In [ ]:
# == 100

In [ ]:
matches_to_one_name = [network_df["fullname"] == match for match in full_matches]

In [ ]:
algorithm_matched_names = network_df[pd.concat(matches_to_one_name, axis=1).sum(axis=1).astype(bool)].loc[:, ["person", "fullname", "report_date", "heading_survey_party", "heading_survey_area"]]
print(algorithm_matched_names.shape)
algorithm_matched_names

### Split aggregated areas + survey parties

In [ ]:
area_to_dissag = [survey for survey in network_df["heading_survey_area"].dropna().unique() if "," in survey]

In [ ]:
area_add_ons = []
for s in area_to_dissag:
    s1, s2 = s.split(", ")
    add_on = network_df.query(f"heading_survey_area == '{s}'").copy()
    area_add_ons.append(len(add_on))
    network_df.loc[network_df["heading_survey_area"] == s, "heading_survey_area"] = s1
    add_on["heading_survey_area"] = s2

    network_df = pd.concat([network_df, add_on])

In [ ]:
network_df["heading_survey_area"].value_counts()

In [ ]:
network_df.shape

In [ ]:
network_df.loc[network_df["heading_survey_area"] == "CENTRAL PROVINCES"].drop_duplicates(subset=["unique_id", "report_date"])

In [ ]:
network_df.to_csv("../data/processed/network.csv", encoding="utf-8-sig")

### Convert to Network

In [ ]:
tidy_network_df = network_df.copy()
tidy_network_df.dropna(subset=["fullname"], inplace=True)

In [ ]:
tidy_network_df.info()

#edges
probably leave military branch as an attribute, it might change over time even if unlikely
medical, acknowledgement, criticism
season
#nodes
survey_party

In [ ]:
# Deal with NaN values for edge attributes
tidy_network_df.loc[tidy_network_df["medical_text"].isna(), "medical_text"] = ""
tidy_network_df.loc[tidy_network_df["role_title"].isna(), "role_title"] = ""
tidy_network_df.loc[tidy_network_df["role_seniority"].isna(), "role_seniority"] = ""
tidy_network_df.loc[tidy_network_df["military_branch_text"].isna(), "military_branch_text"] = ""

# Deal with NaN values for node attributes
tidy_network_df.loc[tidy_network_df["title"].isna(), "title"] = ""
tidy_network_df.loc[tidy_network_df["dateOfDeath"].isna(), "dateOfDeath"] = ""
tidy_network_df["dateOfDeath"] = tidy_network_df["dateOfDeath"].str.replace("/", "-")

In [ ]:
# Create a MultiGraph from the edge list with count and url as edge attributes
G = nx.from_pandas_edgelist(
    tidy_network_df,
    source='unique_id',
    target='heading_survey_area',
    edge_attr=['report_date', 'heading_survey_party', 'leader', 'medical_text', 'role_title', 'role_seniority', 'military_branch_text', "frequency"],
    create_using=nx.Graph()
)
G.nodes

In [ ]:
tidy_network_df["heading_survey_area"].unique()

In [ ]:
person_node_attributes_df["type"].unique()

In [ ]:
person_node_attributes_df = tidy_network_df.dropna(subset="unique_id").set_index("unique_id")[["title", "person", "ethnicity", "dateOfDeath"]]
person_node_attributes_df["type"] = "PERSON"
G.add_nodes_from((n, dict(d)) for n, d in person_node_attributes_df.iterrows())
G.add_nodes_from((n, {"type": "SURVEY"}) for n in tidy_network_df["heading_survey_area"].unique())

In [ ]:
Sigma(G)

In [ ]:
gexf_filepath = f"../data/processed/coleridge_network.gexf"

nx.write_gexf(G, gexf_filepath)